In [ ]:
# Cell 1：掛載 Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content')  # 掛載後立刻回到本機安全位置
print('Drive 掛載完成，目前位置：', os.getcwd())

Mounted at /content/drive
Drive 掛載完成，目前位置： /content


In [ ]:
# Cell 2：Clone darknet（只有第一次需要執行）
import os

os.chdir('/content/drive/MyDrive/Yolov4')
!git clone https://github.com/AlexeyAB/darknet

fatal: destination path 'darknet' already exists and is not an empty directory.


In [ ]:
# Cell 3：啟用 GPU / OpenCV / CUDNN flags
import os

os.chdir('/content/drive/MyDrive/Yolov4/darknet')
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/CUDNN_HALF=0/CUDNN_HALF=1/' Makefile
print('Makefile flags 設定完成')

Makefile flags 設定完成


In [ ]:
# Cell 4：產生 train.txt（訓練圖片清單）
# 請確認 /content/drive/MyDrive/Yolov4/obj/ 已放好 .jpg 和對應的 .txt 標注檔
import os

os.chdir('/content/drive/MyDrive/Yolov4')
!python generate_train.py
print('train.txt 已產生，前 3 筆：')
!head -3 /content/drive/MyDrive/Yolov4/train.txt

/content/drive/MyDrive/Yolov4/obj
train.txt 已產生，前 3 筆：
/content/drive/MyDrive/Yolov4/obj/xy_044_063_08451ee8-5e6e-11f1-aa69-00a5547afa08_jpg.rf.f268e6d049f356aa0924b3927a9ad2a2.jpg
/content/drive/MyDrive/Yolov4/obj/xy_177_063_5d6abda0-5e6f-11f1-aa69-00a5547afa08_jpg.rf.02eba0f399d966089fd142e6d0aa7918.jpg
/content/drive/MyDrive/Yolov4/obj/xy_044_063_e5712478-5e6f-11f1-aa69-00a5547afa08_jpg.rf.6f05060e783dd71c08837bb0a384e913.jpg


In [ ]:
import os

TEST_DIR = '/content/drive/MyDrive/Yolov4/test'

image_files = [
    os.path.join(TEST_DIR, f)
    for f in os.listdir(TEST_DIR)
    if f.endswith('.jpg')
]

with open('/content/drive/MyDrive/Yolov4/test.txt', 'w') as f:
    f.write('\n'.join(image_files) + '\n')

print(f"test.txt 已建立，共 {len(image_files)} 張")
!head -3 /content/drive/MyDrive/Yolov4/test.txt

test.txt 已建立，共 18 張
/content/drive/MyDrive/Yolov4/test/xy_032_072_11500a2a-5e6e-11f1-aa69-00a5547afa08_jpg.rf.cbae312e6adefa3cf9ade48c420f80c3.jpg
/content/drive/MyDrive/Yolov4/test/xy_189_069_c19edb10-5e6c-11f1-aa69-00a5547afa08_jpg.rf.7961fd3a40a9e37ad3161475cd2c42a6.jpg
/content/drive/MyDrive/Yolov4/test/xy_191_073_cc154cb8-5e6d-11f1-aa69-00a5547afa08_jpg.rf.3e48130f4647a5e5df216f987f518216.jpg


In [ ]:
# Cell 5：安裝 OpenCV
!sudo apt update -q
!sudo apt install -y libopencv-dev

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,129 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,703 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/

In [ ]:
# Cell 6：複製到本機 → 修復 Makefile CUDA 路徑 → 編譯
# 在本機 /content/ 編譯比在 Drive 上快 10 倍以上（3~8 分鐘 vs 30~90 分鐘）
import os, shutil

DRIVE_DARKNET = '/content/drive/MyDrive/Yolov4/darknet'
LOCAL_DARKNET = '/content/darknet'

# 複製到本機
if not os.path.exists(LOCAL_DARKNET):
    print('複製 darknet 到本機...')
    shutil.copytree(DRIVE_DARKNET, LOCAL_DARKNET)
    print('複製完成')
else:
    print('本機已有 darknet，略過複製')

# 修復 Makefile CUDA 路徑（舊版 Makefile 的 shell glob 會產生錯誤字串汙染）
makefile_path = f'{LOCAL_DARKNET}/Makefile'
with open(makefile_path, 'r') as f:
    content = f.read()

BAD  = 'ls: cannot access /usr/local/cuda-*: No such file or directory'
GOOD = '/usr/local/cuda-12.8'
fixed = content.replace(BAD, GOOD)

with open(makefile_path, 'w') as f:
    f.write(fixed)

print(f'修復了 {content.count(BAD)} 個 CUDA 路徑汙染點')
print('\n開始編譯（約 3~8 分鐘，有輸出代表正在跑）...')

# 編譯
!cd /content/darknet && make clean && make -j$(nproc) 2>&1 | tail -10

# 備份 binary 回 Drive
!cp /content/darknet/darknet {DRIVE_DARKNET}/darknet
print('\n✅ 編譯完成，binary 已備份至 Drive')

複製 darknet 到本機...
複製完成
修復了 0 個 CUDA 路徑汙染點

開始編譯（約 3~8 分鐘，有輸出代表正在跑）...
rm -rf ./obj/image_opencv.o ./obj/http_stream.o ./obj/gemm.o ./obj/utils.o ./obj/dark_cuda.o ./obj/convolutional_layer.o ./obj/list.o ./obj/image.o ./obj/activations.o ./obj/im2col.o ./obj/col2im.o ./obj/blas.o ./obj/crop_layer.o ./obj/dropout_layer.o ./obj/maxpool_layer.o ./obj/softmax_layer.o ./obj/data.o ./obj/matrix.o ./obj/network.o ./obj/connected_layer.o ./obj/cost_layer.o ./obj/parser.o ./obj/option_list.o ./obj/darknet.o ./obj/detection_layer.o ./obj/captcha.o ./obj/route_layer.o ./obj/writing.o ./obj/box.o ./obj/nightmare.o ./obj/normalization_layer.o ./obj/avgpool_layer.o ./obj/coco.o ./obj/dice.o ./obj/yolo.o ./obj/detector.o ./obj/layer.o ./obj/compare.o ./obj/classifier.o ./obj/local_layer.o ./obj/swag.o ./obj/shortcut_layer.o ./obj/representation_layer.o ./obj/activation_layer.o ./obj/rnn_layer.o ./obj/gru_layer.o ./obj/rnn.o ./obj/rnn_vid.o ./obj/crnn_layer.o ./obj/demo.o ./obj/tag.o ./obj/cifar.o ./o

In [ ]:
# Cell 7：下載預訓練權重
import os

os.chdir('/content/darknet')
!wget -q --show-progress https://github.com/AlexeyAB/darknet/releases/download/yolov4/yolov4-tiny.weights
!wget -q --show-progress https://github.com/AlexeyAB/darknet/releases/download/yolov4/yolov4-tiny.conv.29
!chmod 755 ./darknet
print('權重下載完成')

yolov4-tiny.weights 100%[===================>]  23.13M  60.7MB/s    in 0.4s    
yolov4-tiny.conv.29 100%[===================>]  18.87M  73.3MB/s    in 0.3s    
權重下載完成


In [ ]:
import os, shutil

# obj/ 處理
if os.path.exists('/content/obj'):
    print(f"obj/ 已存在（{len(os.listdir('/content/obj'))} 個檔案），略過")
else:
    print("複製 obj/ ...")
    shutil.copytree('/content/drive/MyDrive/Yolov4/obj', '/content/obj')
    print("obj/ 完成")

# 先看 Drive 有哪些資料夾
print("\nDrive 上的資料夾：")
for item in sorted(os.listdir('/content/drive/MyDrive/Yolov4')):
    full = f'/content/drive/MyDrive/Yolov4/{item}'
    if os.path.isdir(full):
        print(f"  📁 {item}/  ({len(os.listdir(full))} 個檔案)")

複製 obj/ ...
obj/ 完成

Drive 上的資料夾：
  📁 Sign/  (5 個檔案)
  📁 backup/  (2 個檔案)
  📁 darknet/  (40 個檔案)
  📁 obj/  (259 個檔案)
  📁 test/  (36 個檔案)


In [ ]:
import os, shutil

# 複製 test/ 到本機
if os.path.exists('/content/test'):
    print(f"test/ 已存在（{len(os.listdir('/content/test'))} 個檔案）")
else:
    shutil.copytree('/content/drive/MyDrive/Yolov4/test', '/content/test')
    print(f"test/ 複製完成（{len(os.listdir('/content/test'))} 個檔案）")

# 產生 train.txt（本機路徑）
train_files = [
    f'/content/obj/{f}'
    for f in os.listdir('/content/obj')
    if f.endswith('.jpg')
]
with open('/content/drive/MyDrive/Yolov4/train.txt', 'w') as f:
    f.write('\n'.join(train_files) + '\n')

# 產生 test.txt（本機路徑）
test_files = [
    f'/content/test/{f}'
    for f in os.listdir('/content/test')
    if f.endswith('.jpg')
]
with open('/content/drive/MyDrive/Yolov4/test.txt', 'w') as f:
    f.write('\n'.join(test_files) + '\n')

print(f"train: {len(train_files)} 張")
print(f"test:  {len(test_files)} 張")
print("\n前 2 筆 train.txt：")
print('\n'.join(train_files[:2]))

test/ 複製完成（36 個檔案）
train: 129 張
test:  18 張

前 2 筆 train.txt：
/content/obj/xy_191_078_e0c729ba-5e6d-11f1-aa69-00a5547afa08_jpg.rf.c8f13758d4a07db70495918022b1fe97.jpg
/content/obj/xy_039_067_054bd7b8-5e6e-11f1-aa69-00a5547afa08_jpg.rf.e267c0f4310b77c19a0ceec691bc357a.jpg


In [ ]:
import shutil

# 用根目錄的正確 cfg 覆蓋掉 darknet/cfg/ 裡的舊版
shutil.copy(
    '/content/drive/MyDrive/Yolov4/yolov4-tiny-custom.cfg',
    '/content/darknet/cfg/yolov4-tiny-custom.cfg'
)

# 確認已更新
!grep -E 'max_batches|^steps|classes' /content/darknet/cfg/yolov4-tiny-custom.cfg

max_batches = 8000
steps=6400,7200
classes=4
classes=4


In [ ]:
# Cell 8：開始訓練（第一次）
import os

os.chdir('/content/darknet')
!./darknet detector train data/obj.data cfg/yolov4-tiny-custom.cfg yolov4-tiny.conv.29 -dont_show

# 訓練結束後備份權重至 Drive
!cp ./backup/yolov4-tiny_custom_last.weights /content/drive/MyDrive/Yolov4/

串流輸出內容已截斷至最後 5000 行。
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 30 Avg (IOU: 0.978161), count: 1, class_loss = 0.000000, iou_loss = 0.329533, total_loss = 0.329533 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 37 Avg (IOU: 0.945910), count: 3, class_loss = 0.000248, iou_loss = 3.654724, total_loss = 3.654971 
 total_bbox = 757176, rewritten_bbox = 0.005151 % 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 30 Avg (IOU: 0.973687), count: 1, class_loss = 0.000000, iou_loss = 0.324936, total_loss = 0.324936 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 37 Avg (IOU: 0.922064), count: 5, class_loss = 0.000156, iou_loss = 9.906279, total_loss = 9.906434 
 total_bbox = 757182, rewritten_bbox = 0.005151 % 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 30 Avg (IOU: 0.964981), count: 3, class_loss = 0.000017, iou_loss = 0.746916, total_loss = 0.746933 
v3 (iou loss, Normalizer: (iou: 0.07,

In [ ]:
# Cell 9：續訓（若訓練中斷，從 last weights 繼續）
import os

os.chdir('/content/darknet')
!./darknet detector train data/obj.data cfg/yolov4-tiny-custom.cfg backup/yolov4-tiny_custom_last.weights -dont_show

恢復環境

In [ ]:
import os, shutil

# 1. 重新複製 obj.data
shutil.copy(
    '/content/drive/MyDrive/Yolov4/obj.data',
    '/content/darknet/data/obj.data'
)

# 2. 確認訓練圖片還在（如果 /content/obj/ 不見了要重新複製）
if not os.path.exists('/content/obj') or len(os.listdir('/content/obj')) == 0:
    shutil.copytree('/content/drive/MyDrive/Yolov4/obj', '/content/obj')
    print("圖片已重新複製")
else:
    print(f"圖片已存在：{len(os.listdir('/content/obj'))} 張")

# 3. 確認 cfg 正確
!grep 'max_batches' /content/darknet/cfg/yolov4-tiny-custom.cfg

圖片已存在：259 張
max_batches = 8000
